# Lesson 9 — Persistent RAG with sqlitesearch

Query notebook — reads from `faq.db` written by `sqlite-ingest.ipynb`.
No data fetching, no indexing. Just open the DB and query.

In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

from openai import OpenAI
from sqlitesearch import TextSearchIndex
from rag_helper import RAGBase

groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [2]:
# Connect to existing DB — no fit() needed
sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

print(f"Documents in index: {sqlite_index.count()}")

Documents in index: 161


In [3]:
# Direct search — can run even while ingestion notebook is still running
results = sqlite_index.search(
    "Can I still join the course after it started?",
    num_results=5
)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'The homework submission form is still open even though the deadline has passed — can I still submit?',
 'Can I submit homework after the deadline, or get a deadline extension?',
 'I missed the first homework - can I still get a certificate?']

In [4]:
# Swap index into RAGBase — no other code changes needed
# This demonstrates the modular design: search backend is swappable
assistant = RAGBase(
    index=sqlite_index,
    llm_client=groq_client,
)

answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes—you can still join the course even after it has started. If you’d like to earn a certificate, just be sure to submit your capstone project while the submission window is still open (i.e., before the live cohort stops accepting projects).


In [5]:
print(assistant.rag("How do I get a certificate?"))

To earn a certificate you need to:

1. **Be enrolled in a live cohort** – certificates are only issued to students who complete the course during an active (live) cohort. You can study the material on your own, but the final steps must happen while the cohort is open.

2. **Complete the capstone project** – submit a final project that passes the evaluation criteria. Homework isn’t required for the certificate, though it’s recommended.

3. **Finish all assigned peer‑reviews** – after you submit the project you’ll automatically receive peer‑review assignments. Complete those reviews before the peer‑review deadline.

4. **Wait for the batch generation** – once the last project attempt and its peer‑review window close, the instructors generate certificates for the whole cohort. You don’t need to request it.

5. **Check your email** – when the certificate is ready you’ll receive an email (look in spam if you don’t see it). Make sure the name on your profile in the course platform is exactly

In [6]:
sqlite_index.close()